# Depth GINN Well-Constraint Preprocess

Build a model-ready `ginn_well_constraints_v1` NPZ for depth-domain GINN from shifted LAS files and batch synthetic QC metrics.

In [ ]:
from __future__ import annotations

import json
import sys
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path

import lasio
import numpy as np
import pandas as pd

repo_root = Path.cwd().resolve()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent
src_root = repo_root / "src"
if str(src_root) not in sys.path:
    sys.path.insert(0, str(src_root))

from cup.petrel.load import import_well_heads_petrel
from cup.seismic.survey import open_survey
from ginn.well_constraints import WellConstraintBundle, save_well_constraints_npz
from ginn_depth.data import load_lfm_depth_npz

pd.set_option("display.max_columns", 80)


## 1) Configure Inputs

Edit these paths before running. The defaults match the current depth synthetic QC workflow naming.

In [ ]:
data_root = repo_root / "data"

seismic_file = data_root / "raw" / "mero_84_coord_extend"
well_heads_file = data_root / "raw" / "well_heads"
batch_output_dir = data_root / "output_wavelet_batch_synthetic_depth_20260428"
shifted_las_dir = batch_output_dir / "shifted_las"
metrics_path = batch_output_dir / "wavelet_batch_metrics.csv"

ai_lfm_file = data_root / "ai_lfm_depth.npz"
output_dir = batch_output_dir / "ginn_well_constraints"
constraint_path = output_dir / "ginn_depth_well_constraints.npz"
qc_path = output_dir / "ginn_depth_well_constraints_qc.csv"

segy_options = {
    "iline": 5,
    "xline": 21,
    "istep": 1,
    "xstep": 4,
}

confidence_corr_floor = 0.3
confidence_corr_span = 0.4

for path in [seismic_file, well_heads_file, shifted_las_dir, metrics_path, ai_lfm_file]:
    assert path.exists(), f"Missing input: {path}"
output_dir.mkdir(parents=True, exist_ok=True)


## 2) Helper Functions

In [ ]:
def normalize_name(name: object) -> str:
    return str(name).strip()


def find_curve(df: pd.DataFrame, names: tuple[str, ...]) -> str | None:
    normalized = {str(col).strip().upper(): str(col) for col in df.columns}
    for name in names:
        hit = normalized.get(name.upper())
        if hit is not None:
            return hit
    return None


def read_ai_from_shifted_las(path: Path) -> tuple[np.ndarray, np.ndarray, str]:
    las = lasio.read(path)
    df = las.df()
    md = df.index.to_numpy(dtype=float)
    ai_col = find_curve(df, ("AI",))
    if ai_col is not None:
        ai = df[ai_col].to_numpy(dtype=float)
        return md, ai, "AI"

    vp_col = find_curve(df, ("VP_MPS", "VP"))
    rho_col = find_curve(df, ("RHO_GCC", "RHO"))
    if vp_col is None or rho_col is None:
        raise ValueError("LAS must contain AI or both VP_MPS and RHO_GCC curves.")
    ai = df[vp_col].to_numpy(dtype=float) * df[rho_col].to_numpy(dtype=float)
    return md, ai, f"{vp_col}*{rho_col}"


def confidence_from_corr(corr: object) -> float:
    try:
        corr_value = float(corr)
    except (TypeError, ValueError):
        return 0.0
    if not np.isfinite(corr_value):
        return 0.0
    return float(np.clip((corr_value - confidence_corr_floor) / confidence_corr_span, 0.0, 1.0))


def interpolate_ai_to_depth_axis(md: np.ndarray, ai: np.ndarray, kb_m: float, depth_axis_m: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    tvdss = np.asarray(md, dtype=float) - float(kb_m)
    ai = np.asarray(ai, dtype=float)
    valid = np.isfinite(tvdss) & np.isfinite(ai) & (ai > 0.0)
    if int(valid.sum()) < 2:
        raise ValueError("Too few finite positive AI samples after MD->TVDSS conversion.")
    tvdss = tvdss[valid]
    ai = ai[valid]
    order = np.argsort(tvdss)
    tvdss = tvdss[order]
    ai = ai[order]
    unique_tvdss, unique_idx = np.unique(tvdss, return_index=True)
    unique_ai = ai[unique_idx]
    if unique_tvdss.size < 2:
        raise ValueError("Too few unique TVDSS samples.")
    ai_on_axis = np.interp(depth_axis_m, unique_tvdss, unique_ai, left=np.nan, right=np.nan)
    mask = np.isfinite(ai_on_axis) & (ai_on_axis > 0.0)
    if int(mask.sum()) < 2:
        raise ValueError("Too few AI samples overlap the depth LFM axis.")
    return ai_on_axis.astype(np.float32), mask


def nearest_flat_index(ai_lfm, iline: float, xline: float) -> tuple[int, int, int, float, float]:
    il_idx = int(np.argmin(np.abs(ai_lfm.ilines - float(iline))))
    xl_idx = int(np.argmin(np.abs(ai_lfm.xlines - float(xline))))
    flat_idx = il_idx * int(ai_lfm.xlines.size) + xl_idx
    return flat_idx, il_idx, xl_idx, float(ai_lfm.ilines[il_idx]), float(ai_lfm.xlines[xl_idx])


## 3) Load Project Inputs

In [ ]:
ai_lfm = load_lfm_depth_npz(ai_lfm_file)
depth_axis_m = ai_lfm.samples.astype(np.float32)
well_heads_df = import_well_heads_petrel(well_heads_file)
metrics_df = pd.read_csv(metrics_path)
survey = open_survey(seismic_file, seismic_type="segy", segy_options=segy_options)

heads_by_name = {normalize_name(row["Name"]): row for _, row in well_heads_df.iterrows()}
metrics_by_name = {normalize_name(row["well_name"]): row for _, row in metrics_df.iterrows()}

las_paths = sorted(shifted_las_dir.glob("*.las"))
print("Shifted LAS count:", len(las_paths))
print("Depth samples:", depth_axis_m.size, depth_axis_m[0], depth_axis_m[-1])


## 4) Build Constraint Arrays

In [ ]:
records = []
qc_rows = []

for las_path in las_paths:
    well_name = las_path.stem
    qc = {"well_name": well_name, "las_path": str(las_path), "status": "skipped"}
    try:
        head = heads_by_name.get(well_name)
        if head is None:
            raise ValueError("Missing well head row.")
        kb_m = float(head["Well datum value"])
        well_x = float(head["Surface X"])
        well_y = float(head["Surface Y"])
        if not np.isfinite(kb_m) or not np.isfinite(well_x) or not np.isfinite(well_y):
            raise ValueError("Non-finite KB or surface coordinates.")

        iline, xline = survey.coord_to_line(well_x, well_y)
        flat_idx, il_idx, xl_idx, nearest_inline, nearest_xline = nearest_flat_index(ai_lfm, iline, xline)
        md, ai, ai_source = read_ai_from_shifted_las(las_path)
        ai_on_axis, mask = interpolate_ai_to_depth_axis(md, ai, kb_m, depth_axis_m)
        log_ai = np.zeros_like(ai_on_axis, dtype=np.float32)
        log_ai[mask] = np.log(np.clip(ai_on_axis[mask], 1e-6, None)).astype(np.float32)

        metric = metrics_by_name.get(well_name)
        corr = np.nan if metric is None else metric.get("corr", np.nan)
        confidence = confidence_from_corr(corr)
        weight = np.zeros_like(ai_on_axis, dtype=np.float32)
        weight[mask] = confidence

        records.append(
            {
                "well_name": well_name,
                "flat_idx": int(flat_idx),
                "inline": float(nearest_inline),
                "xline": float(nearest_xline),
                "log_ai": log_ai,
                "ai": np.nan_to_num(ai_on_axis, nan=0.0).astype(np.float32),
                "mask": mask.astype(bool),
                "weight": weight,
            }
        )
        qc.update(
            {
                "status": "ok",
                "ai_source": ai_source,
                "kb_m": kb_m,
                "well_x": well_x,
                "well_y": well_y,
                "resolved_inline": float(iline),
                "resolved_xline": float(xline),
                "nearest_inline": float(nearest_inline),
                "nearest_xline": float(nearest_xline),
                "flat_idx": int(flat_idx),
                "corr": float(corr) if np.isfinite(corr) else np.nan,
                "confidence": confidence,
                "valid_samples": int(mask.sum()),
            }
        )
    except Exception as exc:
        qc.update({"status": "failed", "error": str(exc)})
    qc_rows.append(qc)

qc_df = pd.DataFrame(qc_rows)
qc_df.to_csv(qc_path, index=False)
display(qc_df)

if not records:
    raise ValueError("No successful wells; inspect the QC CSV before building constraints.")

duplicates = [flat for flat, count in Counter(row["flat_idx"] for row in records).items() if count > 1]
if duplicates:
    duplicate_wells = [row["well_name"] for row in records if row["flat_idx"] in duplicates]
    raise ValueError(f"Duplicate nearest traces are not supported in v1: {duplicates}, wells={duplicate_wells}")


## 5) Save Model-Ready NPZ

In [ ]:
metadata = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "source_notebook": "ginn_depth_well_constraints_preprocess@20260504.ipynb",
    "shifted_las_dir": str(shifted_las_dir),
    "metrics_path": str(metrics_path),
    "well_heads_file": str(well_heads_file),
    "ai_lfm_file": str(ai_lfm_file),
    "confidence_formula": f"clip((corr - {confidence_corr_floor}) / {confidence_corr_span}, 0, 1)",
    "qc_path": str(qc_path),
    "n_input_las": int(len(las_paths)),
    "n_successful_wells": int(len(records)),
    "status_counts": qc_df["status"].value_counts(dropna=False).to_dict(),
}

bundle = WellConstraintBundle(
    sample_domain="depth",
    sample_unit="m",
    samples=depth_axis_m.astype(np.float32),
    flat_indices=np.array([row["flat_idx"] for row in records], dtype=np.int64),
    well_log_ai_target=np.stack([row["log_ai"] for row in records]).astype(np.float32),
    well_ai_target=np.stack([row["ai"] for row in records]).astype(np.float32),
    well_mask=np.stack([row["mask"] for row in records]).astype(bool),
    well_weight=np.stack([row["weight"] for row in records]).astype(np.float32),
    well_names=np.array([row["well_name"] for row in records]),
    inline=np.array([row["inline"] for row in records], dtype=np.float32),
    xline=np.array([row["xline"] for row in records], dtype=np.float32),
    metadata=metadata,
)

save_well_constraints_npz(constraint_path, bundle)
print("Saved:", constraint_path)
print("QC:", qc_path)
print(json.dumps(metadata, ensure_ascii=False, indent=2))
